# Telecom Network KPI Analysis: SQL

The same synthetic dataset, loaded into SQLite with `src/build_database.py`.
Each query below answers a question about network performance.

In [2]:
import sqlite3
from pathlib import Path

import pandas as pd

conn = sqlite3.connect(Path("..") / "data" / "network.db")


def query(sql: str) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn)

In [3]:
query("""
SELECT s.technology,
       COUNT(*)                         AS hours,
       ROUND(AVG(k.throughput_mbps), 1) AS avg_throughput_mbps,
       ROUND(AVG(k.latency_ms), 1)      AS avg_latency_ms
FROM hourly_kpis AS k
JOIN cell_sites AS s ON s.cell_id = k.cell_id
WHERE k.availability_pct = 100
GROUP BY s.technology;
""")

,technology,hours,avg_throughput_mbps,avg_latency_ms
0,4G,15813,31.4,50.6
1,5G,12943,163.7,16.9


In [4]:
query("""
SELECT s.cell_id, s.area, s.technology,
       SUM(CASE WHEN k.prb_utilization_pct >= 80 THEN 1 ELSE 0 END) AS congested_hours,
       ROUND(AVG(k.latency_ms), 1) AS avg_latency_ms
FROM hourly_kpis AS k
JOIN cell_sites AS s ON s.cell_id = k.cell_id
WHERE k.availability_pct = 100
GROUP BY s.cell_id, s.area, s.technology
ORDER BY congested_hours DESC
LIMIT 10;
""")

,cell_id,area,technology,congested_hours,avg_latency_ms
0,CELL_038,Muwaileh,4G,195,65.4
1,CELL_032,Marina,4G,184,64.8
2,CELL_034,Marina,5G,170,21.7
3,CELL_040,Jebel Ali,5G,169,21.7
4,CELL_026,Khalifa City,4G,154,61.5
5,CELL_018,Corniche,5G,139,20.5
6,CELL_031,Marina,5G,137,20.2
7,CELL_012,Marina,4G,136,58.8
8,CELL_002,Muwaileh,5G,134,20.4
9,CELL_007,Deira,4G,125,57.5


In [5]:
query("""
SELECT cell_id,
       COUNT(*)                         AS outage_hours,
       ROUND(AVG(availability_pct), 1)  AS avg_availability_pct
FROM hourly_kpis
WHERE availability_pct < 100
GROUP BY cell_id
ORDER BY outage_hours DESC
LIMIT 10;
""")

,cell_id,outage_hours,avg_availability_pct
0,CELL_001,10,9.9
1,CELL_011,5,24.9
2,CELL_009,4,2.2
3,CELL_010,4,10.3
4,CELL_020,4,58.3
5,CELL_031,4,14.9
6,CELL_033,4,8.6
7,CELL_030,3,41.8
8,CELL_013,2,43.9
9,CELL_018,2,47.4


In [6]:
query("""
WITH cell_stats AS (
    SELECT s.cell_id, s.area, s.technology,
           ROUND(AVG(k.latency_ms), 1) AS avg_latency_ms
    FROM hourly_kpis AS k
    JOIN cell_sites AS s ON s.cell_id = k.cell_id
    WHERE k.availability_pct = 100
    GROUP BY s.cell_id, s.area, s.technology
),
ranked AS (
    SELECT *,
           RANK() OVER (PARTITION BY technology ORDER BY avg_latency_ms DESC) AS latency_rank
    FROM cell_stats
)
SELECT * FROM ranked
WHERE latency_rank <= 3
ORDER BY technology, latency_rank;
""")

,cell_id,area,technology,avg_latency_ms,latency_rank
0,CELL_038,Muwaileh,4G,65.4,1
1,CELL_032,Marina,4G,64.8,2
2,CELL_026,Khalifa City,4G,61.5,3
3,CELL_034,Marina,5G,21.7,1
4,CELL_040,Jebel Ali,5G,21.7,1
5,CELL_018,Corniche,5G,20.5,3


In [7]:
query("SELECT COUNT(*) AS outage_cell_hours FROM hourly_kpis WHERE availability_pct < 100;")

,outage_cell_hours
0,44


## Cross-check with pandas
The SQL results match the pandas analysis in `01_exploration.ipynb`:
the same most-congested cell (CELL_038, 195 hours) and the same 44 outage cell-hours.